# L2C Review : exploration des données et démo complète (colonnes CLP)

Ce notebook a deux parties.

1. **Explorer les données.** On regarde ce que contient le corpus. On montre pourquoi
   certains PDF se lisent directement et d'autres non. On présente les méthodes de
   lecture qu'on a essayées.
2. **Le pipeline complet pour les colonnes de CLP.** On part du plan de structure, on
   lit le dessin d'atelier (DA), on met les deux dans le même format, on compare, puis
   on écrit le JSON de l'annexe A.

Tout tourne en local sur le vrai corpus. Aucune valeur du corrigé n'est écrite dans le code.

**Pour le lancer.** Faites `install.cmd` (ou `pip install -e ".[app,da,dev,notebook]"`),
ouvrez ce fichier dans Jupyter, puis *Run All*.

**Durée.** La première fois, comptez environ 25 minutes sur un portable. Presque tout ce
temps sert à l'OCR des colonnes du dessin d'atelier (15 à 20 minutes). Le résultat de
l'OCR est gardé dans `out/notebook_colonnes_clp/`. Les fois suivantes, le notebook
tourne en 5 minutes environ.

**Confidentialité.** Les dessins sont confidentiels. Ce notebook est versionné
**sans ses sorties**, car les figures sont des extraits de dessins. Videz les sorties
avant de faire un commit.

In [ ]:
import collections
import copy
import json
import os
import re
import sys
import time
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "l2c").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymupdf
from IPython.display import display
from matplotlib.patches import Circle, Rectangle

from l2c.da.imageread import PageImage

CORPUS = Path(os.environ.get("L2C_CORPUS", Path.home() / "Downloads" / "l2c-participants"))
PROJECTS = ["CLP", "WP2", "LIGREP", "EspCa3B"]
assert CORPUS.is_dir(), f"corpus introuvable dans {CORPUS} : définissez L2C_CORPUS"

pd.set_option("display.max_colwidth", 80)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.titlesize": 10,
                     "axes.titleweight": "bold", "axes.titlelocation": "left"})
BLUE, ORANGE, AQUA, RED = "#2a78d6", "#eb6834", "#1baf7a", "#d03b3b"
INK, MUTED, SURFACE = "#0b0b0b", "#52514e", "#fcfcfb"


def show(src, clip, zoom=2.0, ax=None, title=None):
    """Affiche une zone de la page. Les axes sont en points PDF, comme les annotations."""
    ax = ax or plt.gca()
    ax.imshow(src.render(clip, zoom), extent=(clip.x0, clip.x1, clip.y1, clip.y0))
    ax.set_axis_off()
    if title:
        ax.set_title(title)
    return ax


def box(ax, rect, color, label=None, pad=0.0, lw=1.2, fontsize=6):
    x0, y0, x1, y1 = rect
    ax.add_patch(Rectangle((x0 - pad, y0 - pad), x1 - x0 + 2 * pad, y1 - y0 + 2 * pad,
                           fill=False, edgecolor=color, linewidth=lw))
    if label:
        ax.text(x0 - pad, y0 - pad - 1, label, color=color, fontsize=fontsize,
                ha="left", va="bottom", fontweight="bold")

# Partie 1 : explorer les données

## 1.1 Ce que contient le corpus

Chaque projet a un jeu de plans de structure (`L2C_PLAN_STR_*.pdf`) et un dossier `DA/`.
Ce dossier contient les dessins d'atelier, rangés à la façon du fabricant.

Le module `l2c.da.inventory` parcourt ce dossier. Il devine le type d'élément à partir
des noms de dossiers et de fichiers. Il classe aussi chaque page selon **la façon dont
on peut la lire** : est-ce que `get_text()` retourne des mots, ou est-ce que la page ne
contient que des tracés ?

In [ ]:
from l2c.da.inventory import inventory
from l2c.pipeline import find_plan

started = time.time()
da_files = {p: inventory(str(CORPUS / p)) for p in PROJECTS}

rows = []
for project, files in da_files.items():
    tiers = collections.Counter(t for f in files for t in f.tiers)
    with pymupdf.open(find_plan(str(CORPUS / project))) as doc:
        plan_pages = len(doc)
        plan_text = sum(len(page.get_text("words")) > 20 for page in doc)
    rows.append({"projet": project, "pages du plan": plan_pages,
                 "pages du plan avec texte": plan_text,
                 "fichiers DA": len(files), "pages DA": sum(f.pages for f in files),
                 "pages DA avec texte": tiers[1],
                 "pages DA sans texte": sum(n for tier, n in tiers.items() if tier != 1)})
census = pd.DataFrame(rows).set_index("projet")
census.loc["Total"] = census.sum()
print(f"recensement fait en {time.time() - started:.0f} s")
census

Toutes les pages des plans ont du texte. Pour les dessins d'atelier, ça dépend du
logiciel du fabricant.

In [ ]:
shown = census.drop("Total")
fig, ax = plt.subplots(figsize=(7.5, 2.4), facecolor=SURFACE)
ax.set_facecolor(SURFACE)
left = np.zeros(len(shown))
for column, color in (("pages DA avec texte", BLUE), ("pages DA sans texte", ORANGE)):
    values = shown[column].to_numpy()
    bars = ax.barh(shown.index, values, left=left, height=0.55, color=color,
                   edgecolor=SURFACE, linewidth=2, label=column.replace("pages DA ", ""))
    for bar, value in zip(bars, values):
        if value:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_y() + bar.get_height() / 2,
                    str(value), ha="center", va="center", color="white", fontweight="bold")
    left += values
ax.invert_yaxis()
ax.set_xlabel("pages de dessins d'atelier", color=MUTED)
ax.set_title("Pages de dessins d'atelier par projet : avec ou sans texte", color=INK)
ax.tick_params(colors=MUTED, length=0)
ax.grid(axis="x", color="#e4e3df", linewidth=0.6)
ax.set_axisbelow(True)
for spine in ax.spines.values():
    spine.set_visible(False)
ax.legend(frameon=False, loc="lower right", labelcolor=MUTED)
plt.tight_layout()
plt.show()

La différence se joue par fichier, pas par page. Un PDF est soit tout en texte, soit
sans aucun texte.

Le même type d'élément se retrouve des deux côtés. Voici les dessins de semelles des
quatre projets (`1` = texte, `2` = tracés vectoriels seulement).

In [ ]:
mixed = [f.name for files in da_files.values() for f in files if len(set(f.tiers)) > 1]
print("fichiers qui mélangent des pages avec et sans texte :", len(mixed))

pd.DataFrame([{"projet": project, "dossier": f.folder, "fichier": f.name, "pages": f.pages,
               "type de lecture par page": f.tiers}
              for project, files in da_files.items() for f in files
              if f.type_element == "semelle"])

## 1.2 Le même genre de dessin, deux PDF très différents

On prend deux dessins de semelles : celui de CLP (avec texte) et celui de WP2 (sans
texte). À l'œil, c'est le même genre de document. Pour une librairie PDF, pas du tout.

In [ ]:
CLP_DA = CORPUS / "CLP" / "DA" / "Fondations" / "CLP_SEMELLES FND.pdf"
WP2_DA = CORPUS / "WP2" / "DA" / "Semelles et radiers" / "WP2_SEMELLES-PART1.pdf"


def open_page(path, index):
    doc = pymupdf.open(path)
    page = doc[index]
    if page.rotation:
        page.remove_rotation()      # un seul repère pour le texte, les tracés et les pixels
    return doc, page


clp_doc, clp_page = open_page(CLP_DA, -1)
wp2_doc, wp2_page = open_page(WP2_DA, 0)
clp_src, wp2_src = PageImage(clp_page), PageImage(wp2_page)

pd.DataFrame({name: {"mots retournés par get_text()": len(page.get_text("words")),
                     "polices": len(page.get_fonts()),
                     "images": len(page.get_images()),
                     "tracés vectoriels": len(page.get_drawings())}
              for name, page in (("semelles CLP", clp_page), ("semelles WP2", wp2_page))})

La page de WP2 **n'est pas un scan** : elle ne contient aucune image. Elle ne contient
aucune police non plus. Ses lettres sont des milliers de tracés vectoriels. `get_text()`
n'a donc rien à retourner.

Voici une annotation de chaque dessin, avec ce que `get_text()` donne pour cette zone.
(WP2 dessine sa feuille couchée dans une page en portrait. On tourne l'extrait pour
l'afficher.)

In [ ]:
title_word = next(w for w in clp_page.get_text("words") if w[4] == "NOMENCLATURE")
CLP_CLIP = pymupdf.Rect(title_word[0] - 45, title_word[1] - 6, title_word[0] + 250, title_word[1] + 88)
WP2_CLIP = pymupdf.Rect(630, 1440, 700, 1590)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
show(clp_src, CLP_CLIP, 5, axes[0], "CLP : nomenclature des semelles")
axes[1].imshow(np.rot90(wp2_src.render(WP2_CLIP, 6), -1))
axes[1].set_axis_off()
axes[1].set_title("WP2 : annotation d'une semelle")
plt.tight_layout()
plt.show()

for name, page, clip in (("CLP", clp_page, CLP_CLIP), ("WP2", wp2_page, WP2_CLIP)):
    text = " | ".join(page.get_text("text", clip=clip).split("\n")).strip(" |")
    print(f"{name} get_text() : {text[:330]!r}")

Alors qu'est-ce qu'il y a dans la page de WP2 ? Des contours. Le logiciel de dessin a
transformé les lettres en formes. Chaque caractère est une petite forme pleine.

On trace ci-dessous les formes brutes de l'extrait, une couleur par forme. On voit que
les caractères n'existent que comme de la géométrie.

In [ ]:
def outline(item):
    """Les points d'un morceau de tracé (ligne, courbe, rectangle ou quadrilatère)."""
    if item[0] == "re":
        r = item[1]
        return [r.tl, r.tr, r.br, r.bl, r.tl]
    if item[0] == "qu":
        q = item[1]
        return [q.ul, q.ur, q.lr, q.ll, q.ul]
    return list(item[1:])


paths = [d for d in wp2_page.get_drawings()
         if WP2_CLIP.contains(d["rect"]) and d.get("fill") != (1.0, 1.0, 1.0)]
glyphs = [d for d in paths if d["type"] == "f" and d["rect"].width < 12 and d["rect"].height < 12]
print(f"{len(paths)} tracés dans l'extrait, dont {len(glyphs)} petites formes pleines (les caractères)")

fig, ax = plt.subplots(figsize=(9, 3.4))
palette = plt.get_cmap("tab10").colors
for k, d in enumerate(paths):
    for item in d["items"]:
        points = outline(item)
        # tourné d'un quart de tour, comme l'image du dessus
        ax.plot([-p.y for p in points], [p.x for p in points], color=palette[k % 10], linewidth=0.9)
ax.set_aspect("equal")
ax.invert_yaxis()
ax.set_axis_off()
ax.set_title("Le même extrait de WP2, tracé à partir de ses formes (une couleur par forme)")
plt.show()

## 1.3 Trois façons de lire une page : ce qu'on a essayé

| # | Lecteur | Idée | Résultat |
|---|---|---|---|
| 1 | **Couche de texte** (`get_text`) | on reçoit les caractères exacts et leur position | parfait quand le texte existe : tous les plans, mais seulement 81 pages de DA sur 421 |
| 2 | **Décodeur de formes** (`l2c.da.decode`) | chaque caractère est une forme, on la compare à des modèles de lettres | a marché sur certains fichiers, mais dépend de la façon dont chaque logiciel dessine le texte |
| 3 | **OCR sur l'image** (`l2c.da.imageread`, RapidOCR / PP-OCR, local, sur CPU) | on transforme la page en image et on la lit comme une personne | un seul lecteur pour tous les fabricants, mais il faut connaître la mise en page pour rester précis |

Voici les trois lecteurs sur l'annotation de WP2 vue plus haut :

In [ ]:
from l2c.da.decode import decode_page
from l2c.da.parsers import colonne_clp as ocr_engine      # contient le moteur OCR local partagé

in_crop = lambda line: WP2_CLIP.contains(pymupdf.Point(line.cx, line.cy))

started = time.time()
decoded = [line for line in decode_page(wp2_page) if in_crop(line)]
decode_seconds = time.time() - started

crop = np.ascontiguousarray(np.rot90(wp2_src.render(WP2_CLIP, 6), -1))
crop = cv2.copyMakeBorder(crop, 40, 40, 40, 40, cv2.BORDER_CONSTANT, value=(255, 255, 255))
started = time.time()
read = ocr_engine._ocr()(crop)
ocr_seconds = time.time() - started

print("1. couche de texte    :", repr(wp2_page.get_text("text", clip=WP2_CLIP)))
print(f"2. décodeur de formes : {[line.text for line in decoded]}   (page entière, {decode_seconds:.0f} s)")
print(f"3. OCR sur l'image    : ({ocr_seconds:.1f} s pour l'extrait)")
pd.DataFrame({"texte": read.txts, "confiance": np.round(read.scores, 2)})

Le décodeur compare chaque forme à des modèles de lettres Helvetica droites. Cette
feuille est dessinée couchée, avec une police de logiciel de dessin. Ici, le décodeur
ne sort que du bruit.

Pour le rendre fiable, il faudrait de nouvelles règles pour chaque style de fichier.
Une image, elle, a toujours la même allure, peu importe le logiciel. On a donc choisi
de lire les dessins d'atelier **uniquement par l'image**. On le fait même pour CLP, qui
a pourtant du texte.

L'OCR ramasse parfois un trait du dessin, comme le `--1` ci-dessus. C'est pour ça qu'on
ajoute les vérifications décrites plus bas.

Ce choix donne un avantage. Sur les pages qui ont du texte, ce texte caché devient un
**juge indépendant**. On lit l'image sans regarder le texte. Ensuite on compare avec le
texte que le PDF contient. Voici le résultat sur la nomenclature de CLP :

In [ ]:
from l2c.da.parsers.dalle_clp import read_region

# on compare seulement les mots et les nombres, pour ignorer les espaces autour des tirets
tokens = lambda text: collections.Counter(re.findall(r"[^\W\d_]+|\d+", text))

ocr_lines = read_region(clp_src, CLP_CLIP, 6)
ocr_tokens = tokens(" ".join(line.text for line in ocr_lines))
hidden_tokens = tokens(clp_page.get_text("text", clip=CLP_CLIP))
agree = sum((ocr_tokens & hidden_tokens).values())
print(f"éléments du texte caché aussi lus par l'OCR : {agree} / {sum(hidden_tokens.values())}")
print("dans le texte caché mais pas lu :", dict(hidden_tokens - ocr_tokens))
print("lu mais absent du texte caché   :", dict(ocr_tokens - hidden_tokens))

ax = show(clp_src, CLP_CLIP, 5, plt.subplots(figsize=(9, 2.8))[1],
          "Les zones de texte trouvées par l'OCR dans la nomenclature de CLP")
for line in ocr_lines:
    box(ax, line.rect, BLUE, lw=0.7)
plt.show()

L'OCR brut est bon, mais pas parfait. Les lecteurs ne lui font donc jamais confiance
tout seul :

- **Vocabulaire fermé.** Un diamètre de barre est toujours `10M`, `15M`, `20M`, `25M`,
  `30M`, `35M`, `45M` ou `55M`. Le modèle de données refuse tout le reste. Un `25H` ne
  peut donc pas arriver dans le résultat.
- **Petites zones dont on connaît le sens.** On lit une case de tableau, une bulle
  d'axe ou un marqueur de type, en haute résolution. On ne lit pas toute la feuille
  d'un coup.
- **Ce qui n'est pas lu reste non lu.** Si une valeur est illisible, on le dit et on
  donne la raison. On ne devine pas, et on ne va jamais chercher la valeur dans le plan.

## 1.4 Deux pièges avec les coordonnées

Le livrable demande des positions X/Y en points, à partir du coin en haut à gauche.
Deux particularités des PDF faussent ces positions sans prévenir. Les deux sont
fréquentes dans ce corpus.

In [ ]:
rows = []
for path in sorted(CORPUS.rglob("*.pdf")):
    if path.parent == CORPUS:
        continue                                   # les consignes, pas des dessins
    with pymupdf.open(path) as doc:
        for page in doc:
            rows.append({"projet": path.relative_to(CORPUS).parts[0],
                         "source": "plan" if path.name.startswith("L2C_PLAN_STR") else "DA",
                         "pages": 1,
                         "page tournée (/Rotate)": page.rotation != 0,
                         "origine MediaBox différente de (0,0)":
                             tuple(round(v) for v in page.mediabox)[:2] != (0, 0)})
traps = pd.DataFrame(rows).groupby(["projet", "source"]).sum()
traps.loc[("Total", ""), :] = traps.sum()
traps.astype(int)

- **La rotation.** Sur une page tournée, le texte et les tracés ne sont pas donnés dans
  le même repère. On commence donc toujours par `page.remove_rotation()`.
- **L'origine de la MediaBox.** Certains plans placent leur origine au centre de la
  feuille. PyMuPDF ramène tout en haut à gauche. Les librairies qui retournent les
  coordonnées brutes se trompent d'une demi-feuille.

Le fichier `tests/test_coords.py` surveille ces deux pièges.

---
# Partie 2 : le pipeline complet pour les colonnes de CLP

```text
plan de structure (S-500 à S-505)        dessin d'atelier (CLP_COLONNES Partie 3.pdf)
  texte + géométrie vectorielle            image seulement (OpenCV + OCR local)
  une feuille par niveau                   un grand tableau par page
  une annotation à côté de chaque colonne  une bande verticale par colonne
           │                                        │
           └───────► même ElementRecord ◄───────────┘    une colonne = une position sur la
                           │                             grille + un niveau + ses barres
                    comparaison (code simple)
                           │
              JSON de l'annexe A + liste des différences
```

Les deux documents ne montrent pas les colonnes de la même façon.

- **Le plan** a une feuille par niveau. Sur la vue en plan, chaque colonne a une petite
  annotation : sa section, ses barres verticales (`ARM.`) et ses étriers (`LIG.`).
- **Le dessin d'atelier** est un grand tableau. Chaque bande verticale est une colonne
  du bâtiment. Chaque rangée est un niveau. Une case donne l'armature de cette colonne à
  ce niveau.

Dans les deux cas, une colonne est identifiée par sa position sur la grille (`K-6`) et
son niveau.

## 2.1 Côté plan : du texte et de la géométrie

`run_plan` lit tout le jeu de plans. Il classe chaque feuille grâce à son cartouche,
puis l'envoie au lecteur qui correspond à son type d'élément.

In [ ]:
from l2c.pipeline import run_plan

PROJECT = CORPUS / "CLP"
PLAN_PDF = find_plan(str(PROJECT))

plan = run_plan(PLAN_PDF)
print(f"{plan.totals['sheets_extracted']} feuilles lues en {plan.elapsed_s:.0f} s, "
      f"système d'unités : {plan.unit_system}")
print(dict(collections.Counter(r.type_element for r in plan.records)))

plan_columns = [r for r in plan.records if r.type_element == "colonne"]
column_sheets = [s for s in plan.sheets if s.type_element == "colonne"]
pd.DataFrame([{"feuillet": s.feuillet, "page": s.page, "niveau": s.niveau,
               "annotations COL.": s.diagnostics["callouts"],
               "symboles candidats": s.diagnostics["symbols"],
               "annotations reliées à un symbole": s.diagnostics["assigned"],
               "échelle (points par pouce)": s.diagnostics["scale_pt_per_inch"],
               "colonnes": s.records, "localisées": s.located}
              for s in column_sheets])

Pour chaque feuille, le lecteur fait quatre choses :

1. **Trouver les annotations.** Chaque annotation commence par le mot `COL.`. Les lignes
   en dessous donnent la section, `ARM.` et `LIG.`.
2. **Trouver les symboles.** Une colonne est un petit rectangle plein sur le dessin.
3. **Calculer l'échelle.** L'annotation dit `16"x24"`. Le rectangle a une taille en
   points. En comparant les deux sur toute la feuille, on trouve l'échelle du dessin.
4. **Relier chaque annotation à son symbole.** Le symbole doit avoir la bonne taille et
   être tout près. On résout toute la feuille d'un coup, pour qu'une annotation ne vole
   pas le symbole de sa voisine. Ensuite on place le symbole sur la grille.

Voici un extrait d'une feuille. Le point orange est le centre de l'annotation. Le point
bleu est le symbole de la colonne. Le nom en bleu est la position trouvée sur la grille.

In [ ]:
from l2c.page import open_document, prepare

demo_sheet = next(s for s in column_sheets if s.niveau == "NIVEAU 2")
plan_doc = open_document(PLAN_PDF)
plan_page = prepare(plan_doc, demo_sheet.page - 1, os.path.basename(PLAN_PDF))
plan_src = PageImage(plan_page._page)
sheet_columns = [r for r in plan_columns if r.feuillet == demo_sheet.feuillet]

focus = next(r for r in sheet_columns if r.element == "K-6")
zoom_rect = pymupdf.Rect(focus.x - 330, focus.y - 220, focus.x + 330, focus.y + 220)
ax = show(plan_src, zoom_rect, 3, plt.subplots(figsize=(12, 8))[1],
          f"{demo_sheet.feuillet} ({demo_sheet.niveau}) : annotation (orange), symbole (bleu), position sur la grille")
for r in sheet_columns:
    sx, sy = r.debug.symbol_bbox[:2]
    if not (zoom_rect.contains(pymupdf.Point(r.x, r.y)) and zoom_rect.contains(pymupdf.Point(sx, sy))):
        continue
    ax.plot([r.x, sx], [r.y, sy], color=AQUA, linewidth=1.2)
    ax.plot(r.x, r.y, "o", color=ORANGE, markersize=5)
    ax.plot(sx, sy, "o", color=BLUE, markersize=5)
    ax.text(sx + 6, sy - 6, r.element, color=BLUE, fontsize=9, fontweight="bold")
plt.show()

In [ ]:
def bars_text(armature):
    """Écrit les barres comme sur le plan : 4-25M pour les verticales, 10M@152mm pour les étriers."""
    parts = []
    for b in armature:
        if b.espacement_mm is not None:
            parts.append(f"{b.diametre}@{b.espacement_mm:.0f}mm")
        else:
            parts.append(f"{b.quantite}-{b.diametre}")
    return " · ".join(parts) or "-"


def column_table(records):
    return pd.DataFrame([{"élément": r.element, "niveau": r.debug.niveau,
                          "barres": bars_text(r.armature),
                          "texte lu": " | ".join(r.debug.raw[:3]),
                          "x": round(r.x, 1), "y": round(r.y, 1),
                          "confiance": r.debug.confidence, "lu par": r.debug.decode_path}
                         for r in records])

column_table(sheet_columns).head(8)

## 2.2 Côté dessin d'atelier : l'image seulement

`colonne_clp.parse_page` lit une page du tableau **comme une image**. Il ne lit ni le
texte ni les tracés du PDF, et il ne voit jamais le plan.

On lit deux fichiers :

- `CLP_COLONNES Partie 3.pdf`, au complet. C'est la version à jour.
- la page 5 de `CLP_COLONNES Partie 1.pdf`. Elle contient des colonnes du sous-sol qui
  ne sont pas dans la Partie 3. Les autres pages de la Partie 1 sont d'anciennes
  versions, on ne les lit pas.

**Cette lecture est lente** : environ 3 minutes par page, donc 15 à 20 minutes en tout.
Le notebook garde le résultat dans `out/notebook_colonnes_clp/`. La fois suivante, il le
recharge en une seconde. Mettez `RELIRE = True` pour refaire l'OCR.

On lance le lecteur avec `check=True`. Dans ce mode, il lit d'abord l'image. Ensuite il
note à part ce que dit le texte caché du PDF, pour qu'on puisse comparer plus bas. Le
texte caché ne sert jamais à remplir le résultat.

In [ ]:
from dataclasses import asdict

from l2c.da.parsers import colonne_clp

RELIRE = False
OUT = ROOT / "out" / "notebook_colonnes_clp"
OUT.mkdir(parents=True, exist_ok=True)
COLUMN_DIR = PROJECT / "DA" / "Colonnes"
CURRENT_PDF = COLUMN_DIR / "CLP_COLONNES Partie 3.pdf"
SUPPLEMENT_PDF = COLUMN_DIR / "CLP_COLONNES Partie 1.pdf"


def read_columns(path, pages=None):
    """Lit les pages demandées (toutes par défaut) et garde le résultat sur disque."""
    stat = path.stat()
    stamp = [path.name, stat.st_size, stat.st_mtime_ns, pages]
    cache = OUT / f"cellules_{path.stem}.json"
    if cache.is_file() and not RELIRE:
        saved = json.loads(cache.read_text(encoding="utf-8"))
        if saved["stamp"] == stamp:
            print(f"{path.name} : {len(saved['cells'])} cases rechargées depuis {cache.name}")
            return [colonne_clp.Cell(**cell) for cell in saved["cells"]]
    cells = []
    with pymupdf.open(path) as doc:
        for index in (pages if pages is not None else range(len(doc))):
            page = doc[index]
            if page.rotation:
                page.remove_rotation()
            started = time.time()
            found = colonne_clp.parse_page(page, check=True, max_strips=0)   # 0 = toutes les bandes
            print(f"{path.name} page {index + 1} : {len(found)} cases lues en {time.time() - started:.0f} s")
            cells += found
    cache.write_text(json.dumps({"stamp": stamp, "cells": [asdict(c) for c in cells]},
                                ensure_ascii=False), encoding="utf-8")
    return cells


current_cells = read_columns(CURRENT_PDF)
supplement_cells = read_columns(SUPPLEMENT_PDF, pages=[4])       # la page 5

Regardons maintenant chaque étape sur une page.

**Étape 1 : trouver le tableau.** On cherche les longs traits verticaux qui commencent
et finissent à la même hauteur : ce sont les bords des bandes. Puis on cherche les
traits horizontaux qui traversent presque toute la largeur : ce sont les bords des
rangées.

In [ ]:
demo_cell = next(c for c in current_cells if c.coordinate == "K-6" and c.level == "NIVEAU 2")
da_doc = pymupdf.open(CURRENT_PDF)
da_page = da_doc[demo_cell.page - 1]
if da_page.rotation:
    da_page.remove_rotation()
da_src = PageImage(da_page)
da_rect = da_page.rect

tables = colonne_clp.find_tables(da_page)
fig, ax = plt.subplots(figsize=(12, 8))
show(da_src, da_rect, 0.9, ax, f"Page {demo_cell.page} : les traits du tableau trouvés dans l'image")
for table in tables:
    for x in table.xs:
        ax.plot([x, x], [table.ys[0], table.ys[-1]], color=BLUE, linewidth=1)
    for y in table.ys:
        ax.plot([table.xs[0], table.xs[-1]], [y, y], color=ORANGE, linewidth=1)
    print(f"tableau : {len(table.xs) - 1} bandes (bleu), {len(table.ys) - 1} rangées (orange)")
plt.show()

**Étape 2 : nommer les bandes et les rangées.**

- La case du bas de chaque bande contient la position de la colonne (`K-6`). L'OCR la
  lit. Sur l'image, on l'a recopiée en bleu au-dessus du tableau.
- Le nom du niveau est écrit à gauche du tableau, à la hauteur de chaque rangée. L'OCR
  lit toute cette marge d'un coup.

Les noms de niveaux servent aussi à corriger les rangées. Par exemple, `TRÉFONDS` et
`RADIER` sont des repères de hauteur, pas des étages. Ils ne coupent pas une colonne en
deux. Si un nom de niveau est illisible, on le déduit des niveaux lus au-dessus et en
dessous, et on le marque comme « déduit ».

In [ ]:
page_cells = [c for c in current_cells if c.page == demo_cell.page]
fig, ax = plt.subplots(figsize=(12, 8))
show(da_src, da_rect, 0.9, ax, f"Page {demo_cell.page} : une bande par colonne, une rangée par niveau")
strips, levels = {}, {}
for c in page_cells:
    strips[(c.x0, c.x1)] = c.coordinate
    levels[(c.y0, c.y1)] = c.level + (" (déduit)" if c.storey_inferred else "")
top = min(y0 for y0, _ in levels)
left = min(x0 for x0, _ in strips)
for (x0, x1), coordinate in strips.items():
    ax.text((x0 + x1) / 2, top - 14, coordinate or "?", color=BLUE, fontsize=7,
            ha="center", va="bottom", fontweight="bold", rotation=90)
for (y0, y1), level in levels.items():
    ax.plot([left, max(x1 for _, x1 in strips)], [y1, y1], color=ORANGE, linewidth=0.8)
    ax.text(left - 6, (y0 + y1) / 2, level, color=ORANGE, fontsize=7, ha="right", va="center",
            fontweight="bold")
plt.show()

every = current_cells + supplement_cells
print("bandes sans position lue :", len({(c.page, c.x0) for c in current_cells if not c.coordinate}))
print("cases dont le niveau a été déduit :", sum(c.storey_inferred for c in current_cells),
      "sur", len(current_cells))

**Étape 3 : lire chaque case.** Pour chaque case, l'OCR trouve les lignes de texte. On
relit ensuite chaque ligne en plus grand. Puis une petite grammaire découpe la ligne :

```text
VERT: 4 25M 25Z8-10            4 barres verticales de 25M, repère de fabrication 25Z8-10
ÉTRI: 19 10M 10ET13X21 @6"     19 étriers de 10M, espacés de 6 pouces
```

Si on trouve les verticales sans les étriers (ou l'inverse), l'OCR a sans doute manqué
une ligne. On relit alors cette case seule, avec un zoom plus fort.

In [ ]:
cell_rect = pymupdf.Rect(demo_cell.x0, demo_cell.y0, demo_cell.x1, demo_cell.y1)
view = pymupdf.Rect(cell_rect.x0 - 118, cell_rect.y0 - 12, cell_rect.x1 + 118, cell_rect.y1 + 12)
ax = show(da_src, view, 4, plt.subplots(figsize=(11, 6))[1],
          f"La case {demo_cell.coordinate}, {demo_cell.level} (en orange) et ses voisines")
box(ax, cell_rect, ORANGE, lw=2)
plt.show()

print("lignes lues par l'OCR :")
for line in demo_cell.lines:
    print("   ", line)
print("barres comprises :")
for bar in demo_cell.bars:
    print("   ", bar)
print("résumé :", demo_cell.summary, "| confiance :", demo_cell.confidence)

Une case est gardée seulement si on y trouve **les verticales et les étriers espacés**.
Sinon on l'écarte et on note la raison. La plupart des cases écartées ne sont pas des
erreurs : ce sont des cases sans colonne à ce niveau, ou qui montrent seulement un
croquis de la section.

In [ ]:
def cell_state(c):
    if c.summary:
        return "gardée"
    return {"ink, no text": "dessin sans texte",
            "no verticals and no spaced ties": "texte sans armature de colonne",
            "no spaced ties": "verticales lues, étriers non lus",
            "no verticals": "étriers lus, verticales non lues"}.get(c.reason, c.reason)

states = pd.Series([cell_state(c) for c in current_cells]).value_counts().rename("cases").to_frame()
display(states)

STATE_COLOR = {"gardée": AQUA, "dessin sans texte": "#b9b8b2", "texte sans armature de colonne": ORANGE}
fig, ax = plt.subplots(figsize=(12, 8))
show(da_src, da_rect, 0.9, ax, f"Page {demo_cell.page} : cases gardées (vert), sans armature (orange), sans texte (gris)")
for c in page_cells:
    box(ax, (c.x0, c.y0, c.x1, c.y1), STATE_COLOR.get(cell_state(c), RED), pad=-2, lw=1.4)
plt.show()

**Étape 4 : vérifier la lecture avec le texte caché.** Pour chaque case, on compare le
résumé lu dans l'image avec le résumé qu'on obtient à partir du texte caché du PDF.

In [ ]:
kept = [c for c in current_cells if c.summary]
equal = sum(c.summary == c.oracle_summary for c in kept)
lost = [c for c in current_cells if not c.summary and c.oracle_summary]
print(f"cases gardées dont le résumé est égal à celui du texte caché : {equal} / {len(kept)}")
print(f"cases écartées alors que le texte caché contenait une armature : {len(lost)}")
pd.DataFrame([{"page": c.page, "élément": c.coordinate, "niveau": c.level,
               "lu dans l'image": c.summary or f"écartée : {c.reason}",
               "texte caché": c.oracle_summary}
              for c in current_cells if (c.summary or c.oracle_summary) and c.summary != c.oracle_summary])

Cette étape vérifie ce qui est écrit dans chaque case. Elle ne vérifie pas le nom de la
bande ni celui de la rangée. Pour ça, on a les images plus haut et la comparaison plus
bas.

**Étape 5 : construire la liste finale.** On enlève les répétitions : une même colonne,
au même niveau, avec la même armature, ne compte qu'une fois. Si deux lectures se
contredisent, on garde les deux et on les signale.

On ajoute ensuite les colonnes de la page 5 de la Partie 1, mais seulement celles que la
Partie 3 ne contient pas. La version à jour a toujours priorité.

In [ ]:
current_output, dedup = colonne_clp.clean_output(current_cells, CURRENT_PDF.name)
print(f"Partie 3 : {dedup['input_rows']} cases gardées -> {dedup['output_rows']} colonnes "
      f"({dedup['removed_rows']} répétitions, {len(dedup['conflicts'])} conflits)")

current_records = colonne_clp.records(current_cells, CURRENT_PDF.name)
known = {c.coordinate for c in current_cells if c.coordinate}
supplement_records = [r for r in colonne_clp.records(supplement_cells, SUPPLEMENT_PDF.name)
                      if r.element not in known]
print(f"Partie 1, page 5 : {len(supplement_records)} colonnes de plus")

print(json.dumps(next(row for row in current_output
                      if row["coordinate"] == demo_cell.coordinate and row["storey"] == demo_cell.level),
                 ensure_ascii=False, indent=2))

## 2.3 Un seul format pour les deux côtés

Les deux lecteurs produisent le même `ElementRecord` (annexe A des consignes). Mais les
deux documents ne disent pas la même chose :

- le plan donne une exigence : `4-25M` et `10M@6"` ;
- le dessin d'atelier donne des pièces à fabriquer : `19` étriers, avec un repère et une
  forme.

Comparer le nombre d'étriers n'aurait pas de sens, car le plan ne le donne pas.
`align_records` garde donc la même information des deux côtés :

- **verticales** : quantité et diamètre ;
- **étriers** : diamètre et espacement, en millimètres.

Les détails de fabrication (repères, nombre de pièces, goujons) ne sont pas perdus. Ils
restent dans les données de diagnostic de chaque colonne.

In [ ]:
from l2c.pipeline import ProjectResult
from l2c.record_formats import align_records

da_columns = align_records(current_records + supplement_records)
plan_columns = align_records(plan_columns)

sample = next(r for r in da_columns if r.element == demo_cell.coordinate and r.debug.niveau == demo_cell.level)
print("texte lu dans la case :", sample.debug.raw)
print("atelier :", json.dumps(sample.to_schema(), ensure_ascii=False))
print("plan    :", json.dumps(focus.to_schema(), ensure_ascii=False))

counts = pd.DataFrame({"plan": pd.Series(collections.Counter(r.debug.niveau for r in plan_columns)),
                       "atelier": pd.Series(collections.Counter(r.debug.niveau for r in da_columns))})
counts.loc["Total"] = counts.sum()
counts.fillna(0).astype(int)

## 2.4 La comparaison

`compare` regroupe les colonnes par niveau et par position sur la grille. Ensuite il
associe les barres une à une, selon leur rôle (verticales ou étriers). Le verdict vient
de code simple, pas d'un modèle. Les statuts possibles :

- `same` : identique
- `changed` : armature différente
- `missing_plan` : présent dans le DA seulement
- `missing_da` : présent dans le plan seulement
- `review` : à vérifier

In [ ]:
from l2c.comparison import compare, level

plan_side = ProjectResult("CLP", os.path.basename(PLAN_PDF), plan.unit_system, plan.unit_evidence,
                          plan_columns, column_sheets, plan.elapsed_s)
da_side = ProjectResult("CLP", "DA colonnes", "imperial", {}, da_columns, [], 0.0)

comparison = compare(plan_side, da_side)


def side_text(side):
    return " · ".join(f"{b['diametre']}@{b['espacement_mm']:.0f}mm" if b["espacement_mm"] is not None
                      else f"{b['quantite']}-{b['diametre']}" for b in side) or "-"


table = pd.DataFrame([{"niveau": row["niveau"], "élément": row["element"], "statut": row["status"],
                       "plan": side_text(row["plan"]), "atelier": side_text(row["atelier"]),
                       "position dans le DA": row["atelier_element"],
                       "raison": row["reason"]} for row in comparison])
table["statut"].value_counts().rename("colonnes").to_frame()

In [ ]:
table[table["statut"] != "same"].sort_values(["statut", "niveau", "élément"]).reset_index(drop=True)

Comment lire ces résultats :

- **`changed`** : la même colonne, au même niveau, avec une armature différente. C'est
  ce qu'un réviseur doit regarder en premier. Deux colonnes voisines dont les valeurs
  semblent échangées peuvent aussi venir d'une erreur de position d'un des deux
  lecteurs. Il faut vérifier sur les dessins.
- **`review`** : la paire existe, mais quelque chose reste à confirmer. Souvent, les
  deux lecteurs ont donné des positions proches mais pas identiques (`J-7` et `J.4-7`).
  On associe ces positions voisines, mais on le dit et on demande une vérification.
- **`missing_plan` et `missing_da`** : la colonne existe d'un seul côté. On garde ces
  lignes signalées. On préfère qu'un réviseur écarte une alerte plutôt que l'outil en
  cache une.

**Avec le corrigé.** Le corrigé est lu au moment de l'exécution. Voici ses lignes sur
les colonnes. Pour chacune, on vérifie trois choses : la valeur du plan est extraite,
la valeur de l'atelier est extraite, et la comparaison signale bien la différence.

In [ ]:
from l2c import answer_key

column_sheet_ids = {s.feuillet for s in column_sheets}
key_rows = [row for row in answer_key.load(str(PROJECT / "CLP_dismatch.xlsx"))
            if row.feuillet in column_sheet_ids]
evidence = []
for row in key_rows:
    plan_record = next(r for r in plan_columns
                       if r.feuillet == row.feuillet and r.element == row.localisation)
    da_here = [r for r in da_columns
               if r.element == row.localisation and level(r) == level(plan_record)]
    verdict = next(c for c in comparison
                   if c["element"] == row.localisation and c["niveau"] == level(plan_record))
    evidence.append((row, plan_record, da_here, verdict))
pd.DataFrame([{"feuillet": row.feuillet, "localisation": row.localisation, "niveau": level(plan_record),
               "corrigé : plan": row.plan, "corrigé : atelier": row.atelier,
               "valeur du plan extraite": answer_key.states(plan_record, row.plan, plan.unit_system),
               "valeur de l'atelier extraite": any(answer_key.states(r, row.atelier, plan.unit_system)
                                                   for r in da_here),
               "statut de la comparaison": verdict["status"]}
              for row, plan_record, da_here, verdict in evidence])

Voici la preuve pour chaque ligne du corrigé, côte à côte : l'annotation sur le plan, et
la case du dessin d'atelier.

In [ ]:
plan_sources = {}
fig, axes = plt.subplots(len(evidence), 2, figsize=(12, 4.6 * len(evidence)), squeeze=False)
for (row, plan_record, da_here, verdict), (left_ax, right_ax) in zip(evidence, axes):
    if plan_record.page not in plan_sources:
        plan_sources[plan_record.page] = PageImage(
            prepare(plan_doc, plan_record.page - 1, os.path.basename(PLAN_PDF))._page)
    clip = pymupdf.Rect(plan_record.x - 95, plan_record.y - 45, plan_record.x + 95, plan_record.y + 45)
    show(plan_sources[plan_record.page], clip, 6, left_ax,
         f"Plan {plan_record.feuillet}, {row.localisation}, {level(plan_record)} : {bars_text(plan_record.armature)}")
    da_record = da_here[0]
    x0, y0, x1, y1 = da_record.debug.symbol_bbox
    page = da_doc[da_record.page - 1]
    if page.rotation:
        page.remove_rotation()
    show(PageImage(page), pymupdf.Rect(x0 - 4, y0 - 4, x1 + 4, y1 + 4), 6, right_ax,
         f"Atelier, {row.localisation}, {da_record.debug.niveau} : {bars_text(da_record.armature)}")
plt.tight_layout()
plt.show()

## 2.5 Les livrables

On écrit les colonnes en JSON au format de l'annexe A. On relit ensuite les fichiers
pour les valider. La comparaison complète est écrite à côté. Le dossier `out/` est
ignoré par git.

In [ ]:
from l2c.io_json import validate_file, write_records

for name, records in (("elements_plan.json", plan_columns), ("elements_atelier.json", da_columns)):
    path = OUT / name
    write_records(str(path), records)
    count, errors = validate_file(str(path))
    print(f"{path.relative_to(ROOT)} : {count} éléments, {len(errors)} erreurs de schéma")
(OUT / "comparaison.json").write_text(
    json.dumps(comparison, ensure_ascii=False, indent=2, default=str) + "\n", encoding="utf-8")
print(f"{(OUT / 'comparaison.json').relative_to(ROOT)} : {len(comparison)} lignes")

## Ce que cette démo montre, et ce qu'elle ne montre pas

- **Ce qu'elle montre.** Un type d'élément sur un projet, de deux PDF jusqu'à une liste
  de différences. Les lignes du corrigé sur les colonnes sont retrouvées. On peut
  inspecter chaque étape.
- **Ce qui est propre à ce fabricant.** `colonne_clp` connaît la mise en page de ce
  fabricant : un tableau, une bande par colonne, une rangée par niveau. Un autre
  fabricant demande son propre module de mise en page. Le reste est réutilisé : le
  moteur OCR, la grammaire des barres, le modèle de données et la comparaison.
- **Ce qui n'est pas vérifié de façon indépendante.** Le nom de la bande et celui de la
  rangée de chaque case. Le texte caché confirme le contenu des cases, pas leur
  position. Les lignes `review` et celles présentes d'un seul côté sont les cas où ça
  compte. On les laisse à réviser.
- **Ce qui est lent.** L'OCR des colonnes prend 15 à 20 minutes sur un portable. C'est
  pour ça que le notebook garde le résultat sur disque.
- **Le même pipeline dans l'application.** Le tableau de bord (`ui.cmd`) lance ces
  lecteurs pour les cinq types d'éléments de CLP et montre la même comparaison.